# Madden Live Win-Probability Model Validation

Comparing **Model 1** vs **Model 2** for live win probabilities across 233 Madden matches.

Evaluation dimensions:
- Calibration
- Discrimination (AUC, Brier Score)
- Temporal consistency
- Comparison vs prematch baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load Data

In [ ]:
df = pd.read_csv('case_study_dataset.csv')
print(f'Rows: {len(df):,}  |  Matches: {df["match_id"].nunique()}')
df.head()

## 2. Derive Ground Truth

The home team wins if `score_home > score_away` at the last play of the match.

In [ ]:
# Last play per match => actual outcome
final_scores = (
    df.sort_values(['match_id', 'play_id'])
      .groupby('match_id')[['score_home', 'score_away']]
      .last()
      .assign(home_win=lambda x: (x['score_home'] > x['score_away']).astype(int))
)

df = df.merge(final_scores[['home_win']], on='match_id')
print('Home win rate:', df.drop_duplicates('match_id')['home_win'].mean().round(3))

## 3. Calibration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = {
    'Prematch': 'prematch_home_prob',
    'Model 1':  'home_prob_model_1',
    'Model 2':  'home_prob_model_2',
}

for ax, (label, col) in zip(axes, models.items()):
    prob_true, prob_pred = calibration_curve(df['home_win'], df[col], n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, marker='o', label=label)
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Perfect')
    ax.set_title(f'Calibration — {label}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('calibration.png', bbox_inches='tight')
plt.show()

## 4. Discrimination — AUC & Brier Score

In [ ]:
results = []
for label, col in models.items():
    auc    = roc_auc_score(df['home_win'], df[col])
    brier  = brier_score_loss(df['home_win'], df[col])
    results.append({'Model': label, 'AUC': round(auc, 4), 'Brier Score': round(brier, 4)})

metrics = pd.DataFrame(results).set_index('Model')
print(metrics.to_string())
metrics

## 5. Temporal Consistency

Smooth probability paths within a game are a sign of a well-behaved model.

In [ ]:
# Play-level variance of probability changes within each match
def mean_abs_change(series):
    return series.diff().abs().mean()

temporal = df.sort_values(['match_id', 'play_id']).groupby('match_id').agg(
    m1_mac=('home_prob_model_1', mean_abs_change),
    m2_mac=('home_prob_model_2', mean_abs_change),
)

print('Mean absolute change per play:')
print(f"  Model 1: {temporal['m1_mac'].mean():.4f}")
print(f"  Model 2: {temporal['m2_mac'].mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(temporal['m1_mac'], bins=30, alpha=0.6, label='Model 1')
ax.hist(temporal['m2_mac'], bins=30, alpha=0.6, label='Model 2')
ax.set_xlabel('Mean |Δp| per play')
ax.set_ylabel('Number of matches')
ax.set_title('Temporal Consistency — distribution of probability volatility')
ax.legend()
plt.tight_layout()
plt.savefig('temporal_consistency.png', bbox_inches='tight')
plt.show()

## 6. Sample Game Paths

In [ ]:
sample_ids = df['match_id'].unique()[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, mid in zip(axes.flat, sample_ids):
    g = df[df['match_id'] == mid].sort_values('play_id')
    ax.plot(g['play_id'], g['home_prob_model_1'], label='Model 1')
    ax.plot(g['play_id'], g['home_prob_model_2'], label='Model 2', linestyle='--')
    ax.axhline(g['prematch_home_prob'].iloc[0], color='grey', lw=0.8, linestyle=':', label='Prematch')
    ax.set_title(f'Match {mid}')
    ax.set_xlabel('Play')
    ax.set_ylabel('Home win prob')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=7)

plt.suptitle('Sample game win-probability paths', fontsize=13)
plt.tight_layout()
plt.savefig('sample_paths.png', bbox_inches='tight')
plt.show()

## 7. Summary & Recommendation

In [ ]:
print('=== Model Comparison Summary ===')
print(metrics.to_string())
print()
print('Temporal volatility (lower = smoother):')
print(f"  Model 1: {temporal['m1_mac'].mean():.4f}")
print(f"  Model 2: {temporal['m2_mac'].mean():.4f}")
print()
print('Recommendation: see findings below.')

### Findings

*(Fill in after running the notebook with real data.)*

| Criterion | Model 1 | Model 2 | Winner |
|---|---|---|---|
| Calibration | — | — | — |
| AUC | — | — | — |
| Brier Score | — | — | — |
| Temporal Smoothness | — | — | — |

**Recommended model:** —